# MVR Data Prep

In [9]:
import pandas as pd  
from sklearn.model_selection import train_test_split  
  
# Read your current CSV  
df = pd.read_csv('a4c_mvr_labels.csv')  
  
# Create label mapping for your medical severity levels  
label_mapping = {  
    'trace': 0,  
    'mild': 1,   
    'moderate': 2,  
    'mild_to_moderate': 3,  
    'severe': 4  
}  
  
# Convert string labels to integers  
df['Value_numeric'] = df['Value'].map(label_mapping)  
  
# Verify all labels were mapped correctly  
if df['Value_numeric'].isna().any():  
    print("Warning: Some labels couldn't be mapped!")  
    print("Unmapped labels:", df[df['Value_numeric'].isna()]['Value'].unique())  
  
# First split: separate out test set (60% train+val, 40% test)  
train_val_df, test_df = train_test_split(  
    df,   
    test_size=0.2,  # 20% for test  
    stratify=df['Value_numeric'],   
    random_state=42  
)  
  
# Second split: split train+val into train and validation (64% train, 16% val, 20% test)  
train_df, val_df = train_test_split(  
    train_val_df,  
    test_size=0.2,  # 20% of remaining 80% = 16% of total  
    stratify=train_val_df['Value_numeric'],  
    random_state=42  
)  
  
# Save files with URI and numeric labels (as expected by V-JEPA 2)  
train_df[['URI', 'Value_numeric']].to_csv('a4c_mvr_labels_train.csv', header=False, index=False, sep=' ')  
val_df[['URI', 'Value_numeric']].to_csv('a4c_mvr_labels_val.csv', header=False, index=False, sep=' ')  
test_df[['URI', 'Value_numeric']].to_csv('a4c_mvr_labels_test.csv', header=False, index=False, sep=' ')  
  
# Print split statistics  
print(f"Total samples: {len(df)}")  
print(f"Train: {len(train_df)} ({len(train_df)/len(df)*100:.1f}%)")  
print(f"Validation: {len(val_df)} ({len(val_df)/len(df)*100:.1f}%)")  
print(f"Test: {len(test_df)} ({len(test_df)/len(df)*100:.1f}%)")  
  
# Print label distribution for each split  
print("\nLabel distribution:")  
for split_name, split_df in [("Train", train_df), ("Val", val_df), ("Test", test_df)]:  
    print(f"{split_name}:")  
    for label, count in split_df['Value_numeric'].value_counts().sort_index().items():  
        original_label = [k for k, v in label_mapping.items() if v == label][0]  
        print(f"  {label} ({original_label}): {count}")  
  
# Save the label mapping for reference  
import json  
with open('label_mapping.json', 'w') as f:  
    json.dump(label_mapping, f, indent=2)  
print(f"\nLabel mapping saved to label_mapping.json")

Total samples: 128791
Train: 82425 (64.0%)
Validation: 20607 (16.0%)
Test: 25759 (20.0%)

Label distribution:
Train:
  0 (trace): 49693
  1 (mild): 20761
  2 (moderate): 2899
  3 (mild_to_moderate): 5986
  4 (severe): 3086
Val:
  0 (trace): 12424
  1 (mild): 5190
  2 (moderate): 725
  3 (mild_to_moderate): 1496
  4 (severe): 772
Test:
  0 (trace): 15530
  1 (mild): 6488
  2 (moderate): 906
  3 (mild_to_moderate): 1870
  4 (severe): 965

Label mapping saved to label_mapping.json
